# 第6章：规划（Planning）

## 概念

**规划** = 让 LLM 先制定计划，再按计划执行任务

```
用户任务 → [LLM] → 制定计划 → 按计划执行 → 输出结果
```

## 核心思路

1. **先规划**：把复杂任务拆解成步骤
2. **再执行**：按照计划逐步完成

In [21]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [22]:
# 加载环境变量
load_dotenv()

# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0,
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 第一步：生成计划

让 LLM 为给定主题生成一个结构化的执行计划

In [23]:
# 规划提示词
planning_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是一个专业的规划专家。
你的任务是为给定主题创建一个执行计划。

请输出一个包含3-5个步骤的计划，每个步骤用数字编号。
格式要求：
1. 步骤一：xxx
2. 步骤二：xxx
3. 步骤三：xxx

只输出计划，不要添加其他内容。""",
        ),
        ("user", "请为主题 '{topic}' 创建一个执行计划。"),
    ]
)

# 创建规划链
planning_chain = planning_prompt | llm | StrOutputParser()

# 执行规划
topic = "强化学习在人工智能中的重要性"
plan = planning_chain.invoke({"topic": topic})

print("## 生成的计划 ##")
print(plan)

## 生成的计划 ##
1. 步骤一：研究强化学习的基本原理、核心算法和理论基础。
2. 步骤二：分析强化学习在人工智能中的实际应用案例，如游戏、机器人控制和自动驾驶。
3. 步骤三：评估强化学习对人工智能技术发展的贡献，包括其优势和局限性。
4. 步骤四：探讨强化学习的未来趋势、挑战及其在推动人工智能进步中的重要性。


## 第二步：解析计划

从计划文本中提取每个步骤

In [24]:
# 解析计划中的步骤
def parse_plan_steps(plan_text):
    """从计划文本中提取步骤"""
    # 匹配数字开头的步骤
    steps = re.findall(r"\d+\.\s*(.+)", plan_text)
    return steps


steps = parse_plan_steps(plan)

print("## 解析出的步骤 ##")
for i, step in enumerate(steps, 1):
    print(f"步骤 {i}: {step}")

## 解析出的步骤 ##
步骤 1: 步骤一：研究强化学习的基本原理、核心算法和理论基础。
步骤 2: 步骤二：分析强化学习在人工智能中的实际应用案例，如游戏、机器人控制和自动驾驶。
步骤 3: 步骤三：评估强化学习对人工智能技术发展的贡献，包括其优势和局限性。
步骤 4: 步骤四：探讨强化学习的未来趋势、挑战及其在推动人工智能进步中的重要性。


## 第三步：按计划执行

让 LLM 按照计划的每个步骤分别执行

In [25]:
# 执行提示词
execution_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是一个专业的技术作家。
你的任务是根据给定的步骤，撰写该步骤对应的内容。

要求：
1. 只撰写该步骤的内容，不要扩展
2. 保持简洁，控制在50-100字
3. 使用清晰、专业的语言""",
        ),
        (
            "user",
            """主题：{topic}

当前步骤：{step}

请撰写该步骤的内容。""",
        ),
    ]
)

# 创建执行链
execution_chain = execution_prompt | llm | StrOutputParser()

# 按计划执行每个步骤
step_results = []
for i, step in enumerate(steps, 1):
    print(f"\n## 执行步骤 {i} ##")
    print(f"步骤内容: {step}")
    result = execution_chain.invoke({"topic": topic, "step": step})
    step_results.append(result)
    print(f"执行结果: {result}")


## 执行步骤 1 ##
步骤内容: 步骤一：研究强化学习的基本原理、核心算法和理论基础。
执行结果: 研究强化学习的基本原理，需理解智能体通过与环境交互、依据奖励信号优化决策的核心机制。核心算法包括Q-learning、策略梯度及Actor-Critic等。理论基础涉及马尔可夫决策过程、贝尔曼方程等，为后续应用复杂任务奠定基础。

## 执行步骤 2 ##
步骤内容: 步骤二：分析强化学习在人工智能中的实际应用案例，如游戏、机器人控制和自动驾驶。
执行结果: 强化学习在多个领域展现出强大应用潜力。在游戏领域，AlphaGo通过自我对弈掌握围棋策略；机器人控制中，智能体通过试错学习完成抓取、行走等复杂任务；自动驾驶则利用强化学习优化路径规划与实时决策，应对动态交通环境。这些案例体现了其通过交互学习实现自主决策的核心价值。

## 执行步骤 3 ##
步骤内容: 步骤三：评估强化学习对人工智能技术发展的贡献，包括其优势和局限性。
执行结果: 强化学习通过试错与环境交互的范式，显著提升了人工智能在复杂决策与自适应控制任务中的能力。其核心贡献在于使智能体能在未知环境中自主学习最优策略，典型应用如AlphaGo和自动驾驶。然而，该方法也存在样本效率低、训练过程不稳定、奖励函数设计困难等局限，且在安全关键领域的应用仍需谨慎。

## 执行步骤 4 ##
步骤内容: 步骤四：探讨强化学习的未来趋势、挑战及其在推动人工智能进步中的重要性。
执行结果: 强化学习的未来趋势包括多智能体协作、与深度学习的深度融合以及实时决策应用。其主要挑战涉及样本效率、安全性和可解释性。尽管存在挑战，强化学习仍是推动人工智能向通用智能发展的关键，能够使系统在复杂环境中自主学习和优化决策。


## 第四步：汇总结果

将所有步骤的结果汇总成最终输出

In [26]:
# 汇总提示词
summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是一个专业的编辑。
你的任务是将多个步骤的内容汇总成一篇完整的文章。

要求：
1. 按照步骤顺序组织内容
2. 确保逻辑连贯
3. 添加标题和小标题
4. 使用 Markdown 格式""",
        ),
        (
            "user",
            """主题：{topic}

步骤内容：
{step_contents}

请将以上内容汇总成一篇完整的文章。""",
        ),
    ]
)

# 格式化步骤内容
step_contents = ""
for i, (step, result) in enumerate(zip(steps, step_results), 1):
    step_contents += f"\n步骤 {i}: {step}\n{result}\n"

# 创建汇总链
summary_chain = summary_prompt | llm | StrOutputParser()

# 执行汇总
final_output = summary_chain.invoke({"topic": topic, "step_contents": step_contents})

print("\n\n## 最终输出 ##")
print(final_output)



## 最终输出 ##
# 强化学习：人工智能自主决策的核心引擎

强化学习作为人工智能领域的重要分支，通过智能体与环境的持续交互和试错学习，为复杂决策问题提供了独特的解决范式。本文将从基本原理出发，结合实际应用案例，系统评估其贡献与局限，并展望未来发展趋势，以阐明强化学习在推动人工智能进步中的关键作用。

## 一、强化学习的基本原理与理论基础

强化学习的核心机制在于智能体通过与环境交互，依据获得的奖励信号不断优化其决策策略。这一过程模拟了生物在环境中通过尝试与反馈进行学习的自然方式。

在算法层面，强化学习包含多种核心方法：
- **Q-learning**：通过学习动作价值函数来选择最优动作
- **策略梯度**：直接优化策略参数以最大化累积奖励
- **Actor-Critic**：结合价值函数和策略梯度的优势，提高学习效率

理论基础方面，强化学习建立在**马尔可夫决策过程**的框架之上，利用**贝尔曼方程**描述价值函数的递归关系。这些数学工具为理解和设计强化学习算法提供了坚实的理论支撑，使其能够处理具有随机性和延迟奖励的复杂任务。

## 二、强化学习的实际应用案例

强化学习已在多个领域展现出强大的应用潜力，通过交互学习实现了真正的自主决策能力。

### 游戏领域的突破
DeepMind的**AlphaGo**通过自我对弈的方式，从零开始学习围棋策略，最终击败人类顶尖棋手。这一成就不仅展示了强化学习在复杂策略游戏中的强大能力，也证明了其在处理高维度状态空间和长期规划问题上的有效性。

### 机器人控制的革新
在机器人领域，强化学习使智能体能够通过试错学习完成**抓取、行走、操作**等复杂任务。与传统编程方法相比，强化学习允许机器人在未知环境中自主发现最优控制策略，显著提高了适应性和灵活性。

### 自动驾驶的优化
自动驾驶系统利用强化学习优化**路径规划与实时决策**，能够应对动态变化的交通环境。通过模拟和实际驾驶中的持续学习，系统可以不断改进其在复杂场景下的驾驶策略，提高安全性和效率。

## 三、强化学习的贡献与局限性评估

### 核心贡献
强化学习通过试错与环境交互的范式，显著提升了人工智能在**复杂决策与自适应控制**任务中的能力。其最大贡献在于使智能体能够在未知或动态环境中自主学习最优策略，无需预先编程所有可能的情况。这种能力在

## 完整流程

```
用户: "写一篇关于强化学习的摘要"
    ↓
第一步: 生成计划
  - LLM 分析主题
  - 生成3-5个步骤的计划
    ↓
第二步: 解析计划
  - 提取每个步骤
    ↓
第三步: 按计划执行
  - 对每个步骤分别执行
  - 生成每个步骤的内容
    ↓
第四步: 汇总结果
  - 将所有步骤的内容汇总
  - 生成最终文章
    ↓
输出: 完整的文章
```

## 规划 vs 直接执行

| | 直接执行 | 先规划再执行 |
|---|---|---|
| **方式** | LLM 直接生成回答 | LLM 先制定计划再执行 |
| **优点** | 快速 | 结构清晰、逻辑完整 |
| **缺点** | 可能遗漏要点 | 需要更多步骤 |
| **适用** | 简单任务 | 复杂任务、长文本 |

## 与其他模式的关系

| 第1章 提示链 | 第2章 路由 | 第3章 并行化 | 第4章 反思 | 第5章 工具使用 | 第6章 规划 |
|-------------|-----------|-------------|-----------|--------------|----------|
| A → B → C | 选一条路 | A、B、C 同时执行 | 生成→批评→改进 | LLM + 外部工具 | 计划→执行 |
| 顺序执行 | 选择执行 | 并发执行 | 循环改进 | 扩展能力 | 结构化执行 |